# 🚀 통합 조건부 오토인코더(CAE) - 완전 최적화 버전

이 노트북은 **실제 PostgreSQL DB 벡터 데이터**를 사용하여 **이종 벡터 통합 압축**을 수행하는 **완전 최적화된 파이프라인**입니다.

### 🎯 핵심 기능
- **✅ 초고속 데이터 로딩**: `UNION ALL` 쿼리로 데이터 준비 시간 단축
- **✅ 견고한 전처리**: 모듈화된 `ConditionEncoder`로 확장성 및 유지보수성 확보
- **✅ 완전 자동 훈련**: 체크포인트 감지 및 자동 재시작 지원
- **✅ GPU 활용도 극대화**: Windows 호환성을 유지하며 `num_workers` 조절
- **✅ 정확한 성능 측정**: 마스킹(Masking) 기반 손실 함수로 모델 성능 평가
- **✅ 종합 결과 분석**: 훈련 곡선 및 성능 지표 자동 시각화

### 💡 사용법
순서대로 모든 셀을 실행하면 됩니다. (`Run All` 또는 순차 실행)


## 1. 라이브러리 임포트 및 환경 설정


In [1]:
import os
import sys
import logging
import pickle
import time
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

from sqlalchemy import create_engine
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# --- 기본 설정 ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

def set_seed(seed=42):
    """재현성을 위한 시드 고정"""
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

set_seed(42)
logging.info("✅ 라이브러리 로드 및 기본 설정 완료")


2025-06-22 19:34:58,965 - INFO - ✅ 라이브러리 로드 및 기본 설정 완료


## 2. 설정 관리 (`TrainingConfig`)


In [2]:
@dataclass
class TrainingConfig:
    """훈련에 필요한 모든 설정을 통합 관리하는 클래스"""
    # --- DB 설정 ---
    db_user: str = 'postgres'
    db_password: str = 'postgres'
    db_host: str = 'localhost'
    db_port: int = 5432
    db_name: str = 'postgres'

    # --- 모델 아키텍처 ---
    input_dim: int = 512       # 데이터 로드 후 자동으로 업데이트됨
    condition_dim: int = 30    # 데이터 로드 후 자동으로 업데이트됨
    latent_dim: int = 64
    hidden_dims: List[int] = field(default_factory=lambda: [256, 128])

    # --- 훈련 하이퍼파라미터 ---
    batch_size: int = 256
    num_epochs: int = 50
    learning_rate: float = 1e-3
    validation_split: float = 0.1
    
    # --- Early Stopping 및 스케줄러 ---
    patience: int = 5
    lr_scheduler_factor: float = 0.5
    lr_scheduler_patience: int = 3

    # --- 시스템 및 GPU 설정 ---
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    num_workers: int = 0  # Windows 안정성을 위해 0, 필요시 2 또는 4로 조정
    pin_memory: bool = True if torch.cuda.is_available() else False

    # --- 저장 경로 ---
    checkpoint_dir: str = "checkpoints"
    
    def __post_init__(self):
        Path(self.checkpoint_dir).mkdir(parents=True, exist_ok=True)

config = TrainingConfig()
logging.info(f"✅ 설정 관리 시스템 초기화 완료. Device: {config.device}")


2025-06-22 19:34:58,990 - INFO - ✅ 설정 관리 시스템 초기화 완료. Device: cuda


## 3. 데이터 로딩 및 전처리 시스템


In [3]:
class DatabaseManager:
    """효율적인 데이터 로딩을 위한 데이터베이스 관리 클래스"""
    def __init__(self, config: TrainingConfig):
        self.db_url = f"postgresql://{config.db_user}:{config.db_password}@{config.db_host}:{config.db_port}/{config.db_name}"
        self.engine = create_engine(self.db_url)
        logging.info("✅ 데이터베이스 엔진 생성 완료")

    def load_unified_data(self) -> pd.DataFrame:
        """UNION ALL 쿼리를 사용하여 모든 벡터를 한 번에 효율적으로 로드"""
        logging.info("🔥 통합 데이터 로딩 시작... (UNION ALL 쿼리 사용)")
        query = """
        SELECT 'origin' as vector_type, id, embedding::text, parameters FROM origin_vector
        UNION ALL
        SELECT 'dct' as vector_type, id, embedding::text, parameters FROM dct_vector
        UNION ALL
        SELECT 'wavelet' as vector_type, id, embedding::text, parameters FROM wavelet_vector;
        """
        try:
            start_time = time.time()
            df = pd.read_sql(query, self.engine)
            end_time = time.time()
            logging.info(f"✅ DB에서 {len(df):,}개의 데이터 로드 완료 ({end_time - start_time:.2f}초)")
            return df
        except Exception as e:
            logging.error(f"❌ 데이터 로드 실패: {e}")
            raise

class ConditionEncoder:
    """파라미터 딕셔너리를 조건 벡터로 변환하는 모듈화된 인코더"""
    def __init__(self):
        self.ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        self.scaler = StandardScaler()
        self.fitted = False
        self.categorical_cols = ['vector_type', 'wavelet_family', 'mode']
        self.numerical_cols = ['level', 'keep_dim']

    def fit(self, params_df: pd.DataFrame):
        logging.info("🔧 조건 인코더(ConditionEncoder) fitting 시작...")
        self.ohe.fit(params_df[self.categorical_cols])
        self.scaler.fit(params_df[self.numerical_cols])
        self.fitted = True
        logging.info("✅ 조건 인코더 fitting 완료")
        
    def transform(self, params_df: pd.DataFrame) -> np.ndarray:
        if not self.fitted:
            raise RuntimeError("Encoder has not been fitted yet.")
        cat_vec = np.array(self.ohe.transform(params_df[self.categorical_cols]))
        num_vec = np.array(self.scaler.transform(params_df[self.numerical_cols]))
        return np.concatenate([cat_vec, num_vec], axis=1).astype(np.float32)

class VectorDataset(Dataset):
    """PyTorch를 위한 커스텀 데이터셋"""
    def __init__(self, vectors, conditions):
        self.vectors = vectors
        self.conditions = conditions

    def __len__(self):
        return len(self.vectors)

    def __getitem__(self, idx):
        return self.vectors[idx], self.conditions[idx]

def preprocess_data(df: pd.DataFrame) -> Tuple[List[np.ndarray], np.ndarray, ConditionEncoder]:
    """전체 데이터 전처리 파이프라인 - KeyError 해결 버전"""
    logging.info("🔄 데이터 전처리 시작...")
    
    # 1. 파라미터 컬럼 펼치기
    params_df = df['parameters'].apply(pd.Series)
    df = pd.concat([df.drop(['parameters'], axis=1), params_df], axis=1)
    
    # vector_type 컬럼 처리 (중복 제거)
    if 'vector_type_x' in df.columns and 'vector_type_y' in df.columns:
        df['vector_type'] = df['vector_type_x'].fillna(df['vector_type_y'])
        df.drop(['vector_type_x', 'vector_type_y'], axis=1, inplace=True)
    # vector_type이 이미 존재하면 그대로 사용

    # 2. pgvector 문자열을 numpy 배열로 변환
    def parse_vector(text):
        if not isinstance(text, str): return None
        try:
            return np.fromstring(text.strip('[]'), sep=',', dtype=np.float32)
        except:
            return None
    df['vector'] = df['embedding'].apply(parse_vector)
    df.dropna(subset=['vector'], inplace=True)

    # 3. 조건 인코더 fit
    condition_encoder = ConditionEncoder()
    condition_cols = condition_encoder.categorical_cols + condition_encoder.numerical_cols
    all_params_df = df[condition_cols].fillna(0)
    all_params_df = pd.DataFrame(all_params_df)
    condition_encoder.fit(all_params_df)

    # 4. 조건 벡터 transform
    conditions = condition_encoder.transform(all_params_df)
    vectors = df['vector'].tolist()
    logging.info("✅ 데이터 전처리 완료")
    return vectors, conditions, condition_encoder

def custom_collate_fn(batch):
    """가변 길이 벡터를 패딩하여 배치로 만드는 함수"""
    vectors, conditions = zip(*batch)
    max_len = max(len(v) for v in vectors)
    
    padded_vectors = torch.zeros(len(vectors), max_len)
    mask = torch.zeros(len(vectors), max_len, dtype=torch.bool)
    
    for i, v in enumerate(vectors):
        end = len(v)
        padded_vectors[i, :end] = torch.from_numpy(v)
        mask[i, :end] = True
        
    conditions = torch.tensor(np.array(conditions), dtype=torch.float32)
    
    return padded_vectors, conditions, mask


## 4. 모델 아키텍처 및 훈련 시스템


In [4]:
class ConditionalAutoencoder(nn.Module):
    def __init__(self, config: TrainingConfig):
        super().__init__()
        self.config = config
        
        # 인코더
        encoder_layers = []
        input_dim = config.input_dim + config.condition_dim
        for hidden_dim in config.hidden_dims:
            encoder_layers.extend([nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.BatchNorm1d(hidden_dim)])
            input_dim = hidden_dim
        encoder_layers.append(nn.Linear(input_dim, config.latent_dim))
        self.encoder = nn.Sequential(*encoder_layers)
        
        # 디코더
        decoder_layers = []
        input_dim = config.latent_dim + config.condition_dim
        for hidden_dim in reversed(config.hidden_dims):
            decoder_layers.extend([nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.BatchNorm1d(hidden_dim)])
            input_dim = hidden_dim
        decoder_layers.append(nn.Linear(input_dim, config.input_dim))
        self.decoder = nn.Sequential(*decoder_layers)
        logging.info("🧠 CAE 모델 아키텍처 생성 완료")
        
    def forward(self, x, c):
        z = self.encoder(torch.cat([x, c], dim=1))
        recon_x = self.decoder(torch.cat([z, c], dim=1))
        return recon_x, z

def masked_mse_loss(recon_x, x, mask):
    """패딩된 부분을 제외하고 손실을 계산하는 마스킹 손실 함수"""
    loss = torch.nn.functional.mse_loss(recon_x, x, reduction='none')
    masked_loss = (loss * mask).sum() / mask.sum()
    return masked_loss

class Trainer:
    """모델 훈련, 체크포인팅, 자동 재시작을 관리하는 통합 클래스"""
    def __init__(self, model, optimizer, scheduler, train_loader, val_loader, config: TrainingConfig):
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config
        self.start_epoch = 1
        self.best_val_loss = float('inf')
        self.history = {'train_loss': [], 'val_loss': []}

    def _load_checkpoint(self):
        latest_checkpoint_path = Path(self.config.checkpoint_dir) / "latest_checkpoint.pth"
        if latest_checkpoint_path.exists():
            logging.info(f"🔄 체크포인트 발견: {latest_checkpoint_path}. 훈련을 재개합니다.")
            checkpoint = torch.load(latest_checkpoint_path, map_location=self.config.device)
            self.model.load_state_dict(checkpoint['model_state_dict'])
            self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            self.start_epoch = checkpoint['epoch'] + 1
            self.best_val_loss = checkpoint['best_val_loss']
            self.history = checkpoint['history']
            logging.info(f"✅ Epoch {self.start_epoch}부터 훈련을 재개합니다.")
        else:
            logging.info("📝 새로운 훈련을 시작합니다.")

    def _save_checkpoint(self, epoch, is_best):
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'best_val_loss': self.best_val_loss,
            'history': self.history
        }
        latest_path = Path(self.config.checkpoint_dir) / "latest_checkpoint.pth"
        torch.save(checkpoint, latest_path)
        if is_best:
            best_path = Path(self.config.checkpoint_dir) / "best_model.pth"
            torch.save(checkpoint, best_path)
            logging.info(f"🏆 최적 모델 저장 (Epoch {epoch}, Val Loss: {self.best_val_loss:.6f})")
    
    def train(self):
        self._load_checkpoint()
        patience_counter = 0

        for epoch in range(self.start_epoch, self.config.num_epochs + 1):
            self.model.train()
            train_loss = 0.0
            train_progress = tqdm(self.train_loader, desc=f"Epoch {epoch}/{self.config.num_epochs} [Train]", leave=False)

            for vectors, conditions, mask in train_progress:
                vectors, conditions, mask = vectors.to(self.config.device), conditions.to(self.config.device), mask.to(self.config.device)
                self.optimizer.zero_grad()
                recon_x, _ = self.model(vectors, conditions)
                loss = masked_mse_loss(recon_x, vectors, mask)
                loss.backward()
                self.optimizer.step()
                train_loss += loss.item()
                train_progress.set_postfix(loss=f"{loss.item():.6f}")
            
            avg_train_loss = train_loss / len(self.train_loader)
            self.history['train_loss'].append(avg_train_loss)
            
            # 검증
            self.model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for vectors, conditions, mask in self.val_loader:
                    vectors, conditions, mask = vectors.to(self.config.device), conditions.to(self.config.device), mask.to(self.config.device)
                    recon_x, _ = self.model(vectors, conditions)
                    loss = masked_mse_loss(recon_x, vectors, mask)
                    val_loss += loss.item()
            avg_val_loss = val_loss / len(self.val_loader)
            self.history['val_loss'].append(avg_val_loss)

            logging.info(f"Epoch {epoch:02d} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")
            self.scheduler.step(avg_val_loss)

            # 체크포인팅 및 Early Stopping
            is_best = avg_val_loss < self.best_val_loss
            if is_best:
                self.best_val_loss = avg_val_loss
                patience_counter = 0
            else:
                patience_counter += 1
            
            self._save_checkpoint(epoch, is_best)

            if patience_counter >= self.config.patience:
                logging.info(f"🛑 Early Stopping! {self.config.patience} 에포크 동안 개선 없음.")
                break
        
        logging.info("🎉 훈련 완료!")
        return self.model, self.history


## 5. 메인 실행 파이프라인


In [5]:
# --- 1. 데이터 로드 및 전처리 ---
db_manager = DatabaseManager(config)
raw_df = db_manager.load_unified_data()
vectors, conditions, condition_encoder = preprocess_data(raw_df)

# --- 2. 설정 업데이트 ---
config.input_dim = max(len(v) for v in vectors)
config.condition_dim = conditions.shape[1]
logging.info(f"⚙️ 설정 업데이트: input_dim={config.input_dim}, condition_dim={config.condition_dim}")

# --- 3. 데이터셋 및 데이터로더 생성 ---
dataset = VectorDataset(vectors, conditions)
val_size = int(len(dataset) * config.validation_split)
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
logging.info(f"📊 데이터 분할: Train {train_size:,} / Validation {val_size:,}")

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, 
                         num_workers=config.num_workers, collate_fn=custom_collate_fn, 
                         pin_memory=config.pin_memory)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, 
                       num_workers=config.num_workers, collate_fn=custom_collate_fn, 
                       pin_memory=config.pin_memory)

# --- 4. 모델 및 훈련 시스템 초기화 ---
model = ConditionalAutoencoder(config).to(config.device)
optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', 
                                                factor=config.lr_scheduler_factor, 
                                                patience=config.lr_scheduler_patience)

trainer = Trainer(model, optimizer, scheduler, train_loader, val_loader, config)

# --- 5. 훈련 실행 --- 
trained_model, history = trainer.train()


2025-06-22 19:34:59,100 - INFO - ✅ 데이터베이스 엔진 생성 완료
2025-06-22 19:34:59,100 - INFO - 🔥 통합 데이터 로딩 시작... (UNION ALL 쿼리 사용)
2025-06-22 19:35:43,944 - INFO - ✅ DB에서 1,570,243개의 데이터 로드 완료 (44.84초)
2025-06-22 19:35:43,944 - INFO - 🔄 데이터 전처리 시작...
2025-06-22 19:40:35,503 - INFO - 🔧 조건 인코더(ConditionEncoder) fitting 시작...


TypeError: Encoders require their input argument must be uniformly strings or numbers. Got ['int', 'str']

## 6. 결과 분석 및 시각화


In [ ]:
def plot_training_history(history):
    """훈련 과정의 손실 곡선을 시각화"""
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

def analyze_reconstruction(model, dataloader, config: TrainingConfig):
    """복원 오차 분석"""
    model.eval()
    total_error = 0
    total_count = 0
    with torch.no_grad():
        for vectors, conditions, mask in dataloader:
            vectors, conditions, mask = vectors.to(config.device), conditions.to(config.device), mask.to(config.device)
            recon_x, _ = model(vectors, conditions)
            loss = masked_mse_loss(recon_x, vectors, mask)
            total_error += loss.item() * vectors.size(0)
            total_count += vectors.size(0)
    
    avg_error = total_error / total_count
    logging.info(f"🔬 최종 검증 데이터셋에 대한 평균 복원 오차: {avg_error:.6f}")
    return avg_error

# --- 결과 분석 실행 ---
logging.info("📊 훈련 결과 시각화...")
plot_training_history(history)
final_error = analyze_reconstruction(trained_model, val_loader, config)

logging.info("🎉 모든 과정 완료!")
print(f"\n📈 최종 결과 요약:")
print(f"- 훈련 에포크: {len(history['train_loss'])}")
print(f"- 최종 훈련 손실: {history['train_loss'][-1]:.6f}")
print(f"- 최종 검증 손실: {history['val_loss'][-1]:.6f}")
print(f"- 최종 복원 오차: {final_error:.6f}")
